In [ ]:
import numpy as np
from numpy import pi
from scipy.signal import firwin, lfilter
from functools import lru_cache

def _analytic_qmf(N: int = 16, fc: float = 0.4):
    """Return analytic low/high quadrature‑mirror FIR pair (length N+1)."""
    base = firwin(N + 1, fc)                  # real prototype
    phase = np.exp(2j * np.pi * 0.125 * np.arange(N + 1))
    h = base * phase                          # analytic low‑pass
    g = base[::-1] * (-1.0) ** np.arange(N + 1) * phase[::-1]  # analytic high‑pass
    return h.astype(np.complex128), g.astype(np.complex128)


def _ternary_filters(N=24, fc=0.4):
    """Low‑pass triplet for 3‑way split (Antoni 2007, Sect. 3.2)."""
    h1 = firwin(N+1, 2/3*fc) * np.exp(2j*pi*np.arange(N+1)*0.25/3)
    h2 = h1 * np.exp(2j*pi*np.arange(N+1)/6)
    h3 = h1 * np.exp(2j*pi*np.arange(N+1)/3)
    return h1, h2, h3

def _kurtosis(x):
    x = x - x.mean()
    m2 = np.mean(np.abs(x)**2)
    if m2 == 0:
        return 0.0
    m4 = np.mean(np.abs(x)**4)
    return m4/m2**2 - 2.0

@lru_cache(maxsize=None)
def _binary_path(index, depth):
    """Return vector of 0/1 decisions for a given node index at 'depth'."""
    return np.array([(index >> k) & 1 for k in range(depth)][::-1])

def _dbfb(x, h, g):
    """One level of dyadic (binary) filter bank."""
    a = lfilter(h, 1.0, x)[1::2]
    d = lfilter(g, 1.0, x)[1::2]
    d *= (-1)**np.arange(1, d.size+1)
    return a, d

def _tbfb(x, h1, h2, h3):
    """One level of ternary split (decimate-by-3)."""
    N  = x.size
    a1 = lfilter(h1, 1.0, x)[2:N:3]
    a2 = lfilter(h2, 1.0, x)[2:N:3]
    a3 = lfilter(h3, 1.0, x)[2:N:3]
    return a1, a2, a3

def fast_kurtogram(x, fs, nlevel=7):
    """
    Compute the fast-kurtogram matrix Kwav and return
    (Kwav, level_grid, freq_grid, fc_opt, bw_opt).
    """
    x   = np.asarray(x).ravel().astype(float)
    x  -= x.mean()
    if nlevel > np.log2(x.size) - 7:
        raise ValueError("Signal too short for the requested depth")

    h, g       = _analytic_qmf()
    h1, h2, h3 = _ternary_filters()
    # storage for kurtosis values
    Kwav = np.zeros((2*nlevel, 3*2**nlevel))

    def recurse(sig, depth, row_offset, col_offset, bw):
        """Depth-first traversal of binary‑ternary tree."""
        if depth == 0:
            return
        a, d = _dbfb(sig, h, g)
        for idx, branch in enumerate((a, d)):
            k_val = _kurtosis(branch[len(h):])
            Kwav[row_offset, col_offset + idx*(3*2**(depth-1)):(col_offset + (idx+1)*(3*2**(depth-1)))] = k_val
            # ternary refinement on branch with higher kurtosis
            if idx == np.argmax((_kurtosis(a), _kurtosis(d))):
                a1, a2, a3 = _tbfb(branch, h1, h2, h3)
                for j, sub in enumerate((a1, a2, a3)):
                    sub_k = _kurtosis(sub[len(h1):])
                    Kwav[row_offset+1, col_offset + idx*(3*2**(depth-1)) + j*2**(depth-1):
                                        col_offset + idx*(3*2**(depth-1)) + (j+1)*2**(depth-1)] = sub_k
            recurse(branch, depth-1, row_offset+2, col_offset + idx*(3*2**(depth-1)), bw/2)

    recurse(x, nlevel, 0, 0, fs/2)
    # locate optimum
    ridx, cidx = np.unravel_index(np.argmax(Kwav), Kwav.shape)
    level_grid = np.concatenate(([0],
                    np.sort(np.vstack((np.arange(1, nlevel+1),
                                       np.arange(1, nlevel+1)+np.log2(3)-1)).ravel())[:2*nlevel-1]))
    freq_grid  = fs*(np.arange(3*2**nlevel)/(3*2**(nlevel+1)) + 1/(3*2**(nlevel+2)))
    fc_opt     = freq_grid[cidx]
    bw_opt     = fs*2**(-(level_grid[ridx]+1))
    return Kwav, level_grid, freq_grid, fc_opt, bw_opt


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from brf_lightning.data.bearing.utils import load_vibration_data

# directory to dump the PNGs
out_dir = Path("notebooks/thesis/plots/kurtogram")
out_dir.mkdir(exist_ok=True, parents=True)

results = []                       # store summary rows here

def analyse_file(fp: Path):
    """Load one CSV, compute kurtogram, save plot, return summary dict."""
    # 1 ───── load vibration trace
    df, fs = load_vibration_data(fp)
    x = df["sensor_bearing"].to_numpy(np.float64)  # float64 for numerical stability
    x -= x.mean()

    # 2 ───── kurtogram
    Kwav, levels, freqs, fc, bw = fast_kurtogram(x, fs, nlevel=7)   # ≈ order‑16 filters

    # 3 ───── plot & save
    fig, ax = plt.subplots(figsize=(7, 4))
    im = ax.imshow(
        Kwav,
        origin="upper",
        aspect="auto",
        extent=[freqs[0], freqs[-1], levels[-1], levels[0]],
        cmap="hot",
    )
    lvl_opt = levels[np.argmax(Kwav, axis=1).argmax()]
    ax.scatter(fc, lvl_opt, facecolors="none", edgecolors="cyan", s=60)
    ax.set(
        xlabel="Frequency [Hz]",
        ylabel="Filter‑bank level $k$",
        title=f"{fp.stem} — $f_c$={fc:.1f} Hz, BW≈{bw:.1f} Hz",
    )
    fig.colorbar(im, ax=ax, label="Spectral kurtosis")
    fig.tight_layout()
    fig.savefig(out_dir / f"{fp.stem}_kurtogram.png", dpi=300)
    plt.close(fig)

    # 4 ───── return summary
    return dict(file=fp.name, fs=fs, fc=fc, bw=bw, level=lvl_opt)

# ────────────────────────────────────────────────────────────────────────
for fp in Path("data/bearing_thl_v1").glob("*.csv"):
    res = analyse_file(fp)
    results.append(res)

summary = pd.DataFrame(results).sort_values("file")
print(summary)          # optional: inspect in console


         file      fs           fc           bw     level
3  fAII10.csv  8192.0  2053.333333  4096.000000  2.584963
8  fAII20.csv  8192.0  2053.333333  1365.333333  7.000000
5  fAII30.csv  8192.0  2053.333333  4096.000000  3.000000
7  fIII10.csv  8192.0  3077.333333   682.666667  1.000000
2  fIII20.csv  8192.0     5.333333    32.000000  1.000000
4  fIII30.csv  8192.0  3077.333333  1365.333333  6.000000
1     n10.csv  8192.0  3077.333333  1365.333333  6.584963
0     n20.csv  8192.0  3077.333333   682.666667  6.000000
6     n30.csv  8192.0  1029.333333   341.333333  1.584963


In [ ]:
import numpy as np, matplotlib.pyplot as plt, matplotlib as mpl
from scipy.signal import butter, sosfiltfilt, hilbert, detrend, get_window
from pathlib import Path
from brf_lightning.data.bearing.utils import load_vibration_data


mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex":    True,
    "font.family":    "serif",
    "font.serif":     ["Times"],
    "font.size":      10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize":9,
    "ytick.labelsize":9,
    "pgf.rcfonts":   False,
})

BANDS = {"fAII10": (2053.3,1024),"fAII20":(2053.3,512),"fAII30":(2053.3,512),
         "fIII10": (3077.3, 683),"fIII20":(3077.3,512),"fIII30":(3077.3,1365),
         "n10": (3077.3,1365),"n20":(3077.3,682.7),"n30":(1029.3,341.3)}
SHAFT_HZ = {"fAII10":12,"fAII20":20,"fAII30":30,
            "fIII10":12,"fIII20":20,"fIII30":30,
            "n10":10,"n20":20,"n30":30}
DEFECT_ORDERS = {"outer":[3.05], "inner":[3.95,4.95,5.95], "normal":[]}

# xlim
XMAX = 10

def band_envelope(x, fs, fc, bw, order=6):
    sos = butter(order, [(fc-bw/2)/(fs/2), (fc+bw/2)/(fs/2)],
                 btype="bandpass", output="sos")
    return np.abs(hilbert(sosfiltfilt(sos, detrend(x, type="constant"))))

def order_spectrum(env, fs, fshaft, nfft=65536, win="hann"):
    w = get_window(win, env.size, fftbins=True)
    spec = np.abs(np.fft.rfft(env*w, nfft))
    freq = np.fft.rfftfreq(nfft, 1/fs)
    return freq/fshaft, spec

out_dir = Path("notebooks/thesis/plots/envelope_order_spectra"); out_dir.mkdir(parents=True, exist_ok=True)

for fp in Path("data/bearing_thl_v1").glob("*.csv"):
    stem = fp.stem
    if stem not in BANDS: 
        continue

    df, fs = load_vibration_data(fp)
    fc, bw  = BANDS[stem];  fshaft = SHAFT_HZ[stem]
    env     = band_envelope(df["sensor_bearing"].to_numpy(np.float64), fs, fc, bw)
    orders, amp = order_spectrum(env, fs, fshaft)
    SdB     = 20*np.log10(amp/amp.max() + 1e-12)

    fig, ax = plt.subplots(figsize=(3.3,1.8))
    ax.plot(orders, SdB, lw=.8, color="steelblue")
    ax.set(xlim=(0, XMAX), ylim=(-100, 0),
           xlabel=r"Order ($f/f_{\mathrm{rot}}$)",
           ylabel=r"Envelope [dB rel.\ peak]")

    if stem.startswith("fAII"):
        marks, subtitle = DEFECT_ORDERS["outer"], r"\textbf{Outer Ring}"
    elif stem.startswith("fIII"):
        marks, subtitle = DEFECT_ORDERS["inner"], r"\textbf{Inner Ring}"
    else:
        marks, subtitle = DEFECT_ORDERS["normal"], r"\textbf{Healthy}"

    y_lab = ax.get_ylim()[1] - 2
    for o in marks:
        if o > XMAX: continue
        ax.axvline(o, ls="--", lw=.9, color="crimson", alpha=.35)
        ax.text(o, y_lab, f"{o:.2f}", ha="center", va="top",
                fontsize=9, color="crimson")

    ax.set_title(fr"{subtitle} -- {stem} "
                 fr"\,|\, {fc-bw/2:.0f}–{fc+bw/2:.0f}\,Hz", pad=4)
    ax.grid(ls=":", color="0.75")
    fig.tight_layout(pad=0.3)

    # save TeX‑friendly PGF + PNG preview
    fig.savefig(out_dir / f"{stem}.pgf")
    fig.savefig(out_dir / f"{stem}.png", dpi=300)
    plt.close(fig)
